In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/online_retail_II.csv")

df.shape

(1067371, 8)

In [2]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [3]:
df = df.dropna(subset=["Customer ID"])

df.shape

(824364, 8)

In [4]:
df[df["Invoice"].astype(str).str.startswith("C")].shape

(18744, 8)

In [5]:
df = df[~df["Invoice"].astype(str).str.startswith("C")]

df.shape

(805620, 8)

In [6]:
print("Negative or Zero Quantity:",
      (df["Quantity"] <= 0).sum())

print("Negative or Zero Price:",
      (df["Price"] <= 0).sum())

Negative or Zero Quantity: 0
Negative or Zero Price: 71


In [7]:
df = df[df["Price"] > 0]

df.shape

(805549, 8)

In [8]:
df.to_csv(
    "../data/processed/cleaned_retail_data.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [9]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

In [10]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

snapshot_date

Timestamp('2011-12-10 12:50:00')

In [12]:
df["Revenue"] = df["Quantity"] * df["Price"]

In [13]:
df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [14]:
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice": "nunique",
    "Revenue": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]

rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,12,77556.46
12347.0,2,8,5633.32
12348.0,75,5,2019.40
12349.0,19,4,4428.69
12350.0,310,1,334.40


In [15]:
rfm.shape

(5878, 3)

In [16]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1,2,3,4,5]
)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
Customer ID,,,,,,
12346.0,326,12,77556.46,2,5,5
12347.0,2,8,5633.32,5,4,5
12348.0,75,5,2019.40,3,4,4
12349.0,19,4,4428.69,5,3,5
12350.0,310,1,334.40,2,1,2


In [17]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
Customer ID,,,,,,,
12346.0,326,12,77556.46,2,5,5,255
12347.0,2,8,5633.32,5,4,5,545
12348.0,75,5,2019.40,3,4,4,344
12349.0,19,4,4428.69,5,3,5,535
12350.0,310,1,334.40,2,1,2,212


In [18]:
rfm["RFM_Score"].value_counts().head(10)

RFM_Score
555    474
111    320
455    249
121    173
344    167
211    162
112    149
444    147
122    137
544    134
Name: count, dtype: int64

In [19]:
vip_customers = rfm[rfm["RFM_Score"] == "555"]

vip_customers.shape

(474, 7)

In [20]:
rfm.to_csv(
    "../data/processed/rfm_customer_data.csv"
)

print("RFM dataset saved successfully!")

RFM dataset saved successfully!
